# Notebook 02 — Data Cleaning & ETL Pipeline
**NST DVA Capstone 2 | Healthcare Sector**

> **Purpose:** Ingest the raw diabetic dataset, apply a fully documented, reproducible cleaning pipeline, and export analysis-ready data to `data/processed/`.
>
> Every transformation is logged with a justification. The raw file is never modified.

---
**Pipeline Stage:** (Extract) → **Clean & Transform** → (Analyze) → (Load)

---

## 1. Environment Setup

In [ ]:
# ── Install if on Google Colab ────────────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded.')

## 2. Load Raw Dataset

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_PATH       = '../data/raw/diabetic_data.csv'       # ← update if needed
PROCESSED_PATH = '../data/processed/diabetic_data_clean.csv'
LOG_PATH       = '../data/processed/02_cleaning_log.txt'

os.makedirs('../data/processed', exist_ok=True)

# Load — treat '?' as NaN (standard for this UCI dataset)
df = pd.read_csv(RAW_PATH, na_values=['?', 'Unknown', 'None', ''])

raw_rows, raw_cols = df.shape
print(f'✅ Raw data loaded: {raw_rows:,} rows × {raw_cols} columns')

# ── Transformation log ────────────────────────────────────────────────────────
# We accumulate every step here and write to file at the end
log = []
def record(step, description, before=None, after=None):
    entry = f'[STEP {step:02d}] {description}'
    if before is not None and after is not None:
        entry += f'  |  rows: {before:,} → {after:,}  (removed {before - after:,})'
    log.append(entry)
    print(entry)

record(0, f'Raw dataset loaded: {raw_rows:,} rows × {raw_cols} cols')

## 3. Step 1 — Drop Irrelevant / Administrative Columns

**Justification:** `examide` and `citoglipton` contain only one unique value ("No") and contribute zero information. `weight` has ≈97% missingness and cannot be reliably imputed — dropping it.

In [ ]:
# ── Drop zero-variance & near-empty columns ───────────────────────────────────
COLS_TO_DROP = []

# 1a. Zero-variance medication columns
zero_var = [c for c in df.columns if df[c].nunique(dropna=True) == 1]
COLS_TO_DROP.extend(zero_var)
print(f'Zero-variance columns: {zero_var}')

# 1b. weight — 97%+ missing, not imputable at scale
if 'weight' in df.columns:
    miss_pct = df['weight'].isnull().mean()
    if miss_pct > 0.90:
        COLS_TO_DROP.append('weight')
        print(f'weight: {miss_pct:.1%} missing → dropped')

COLS_TO_DROP = list(set(COLS_TO_DROP))  # deduplicate
df.drop(columns=COLS_TO_DROP, inplace=True, errors='ignore')

record(1, f'Dropped {len(COLS_TO_DROP)} uninformative/near-empty columns: {COLS_TO_DROP}')
print(f'Shape after step 1: {df.shape}')

## 4. Step 2 — Remove Out-of-Scope Records

**Justification:** Readmission analysis is not applicable to patients who died or were transferred to hospice during the encounter. Removing them prevents misleading KPIs.

In [ ]:
# ── Remove deceased / hospice discharges ─────────────────────────────────────
# ICD discharge codes: 11=Expired, 13=Hospice/medical, 14=Hospice/home,
#                      19=Expired at home, 20=Expired in hospital, 21=Expired-unknown
EXPIRED_CODES = [11, 13, 14, 19, 20, 21]

before = len(df)
if 'discharge_disposition_id' in df.columns:
    df = df[~df['discharge_disposition_id'].isin(EXPIRED_CODES)]

record(2, 'Removed expired / hospice discharge encounters', before, len(df))

## 5. Step 3 — Handle Duplicate Encounters

**Justification:** If a patient has multiple encounters, we keep only the first per patient to avoid data leakage in readmission analysis (the readmission IS the second encounter).

In [ ]:
# ── Keep first encounter per patient ─────────────────────────────────────────
before = len(df)
if 'patient_nbr' in df.columns:
    df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')

record(3, 'Kept first encounter per patient_nbr (encounter-deduplication)', before, len(df))
print(f'Unique patients after dedup: {df["patient_nbr"].nunique():,}')

## 6. Step 4 — Handle Missing Values

In [ ]:
# ── 4a. race — impute with mode ───────────────────────────────────────────────
if 'race' in df.columns:
    race_mode = df['race'].mode()[0]
    df['race'].fillna(race_mode, inplace=True)
    print(f'race: missing values filled with mode "{race_mode}"')
    record(4, f'race: NaN → mode "{race_mode}"')

# ── 4b. payer_code — fill with "Unknown" ──────────────────────────────────────
if 'payer_code' in df.columns:
    df['payer_code'].fillna('Unknown', inplace=True)
    print('payer_code: missing → "Unknown"')
    record(5, 'payer_code: NaN → "Unknown"')

# ── 4c. medical_specialty — fill with "Unknown" ───────────────────────────────
if 'medical_specialty' in df.columns:
    df['medical_specialty'].fillna('Unknown', inplace=True)
    print('medical_specialty: missing → "Unknown"')
    record(6, 'medical_specialty: NaN → "Unknown"')

# ── 4d. diag_1/2/3 — fill missing with '0' (no recorded diagnosis) ────────────
for d_col in ['diag_1', 'diag_2', 'diag_3']:
    if d_col in df.columns:
        df[d_col].fillna('0', inplace=True)
print('diag_1/2/3: remaining NaN → "0" (no recorded code)')
record(7, 'diag_1/2/3: NaN → "0"')

# ── 4e. Remaining numeric NaNs — fill with median ────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns
for c in num_cols:
    if df[c].isnull().sum() > 0:
        med = df[c].median()
        df[c].fillna(med, inplace=True)
        print(f'{c}: NaN → median ({med:.2f})')
record(8, 'Remaining numeric NaN → column median')

# ── Verify no missing remain ──────────────────────────────────────────────────
remaining_null = df.isnull().sum().sum()
print(f'\nTotal remaining nulls: {remaining_null}')
assert remaining_null == 0, '⚠️  Still has missing values — check above steps!'
print('✅ No missing values remain.')

## 7. Step 5 — Age Bracket → Numeric Midpoint

**Justification:** Age is stored as `[70-80)` strings. Converting to numeric midpoints (e.g. 75) enables correlation, regression, and numeric binning.

In [ ]:
# ── Age bracket → midpoint ────────────────────────────────────────────────────
AGE_MAP = {
    '[0-10)'  :  5,
    '[10-20)' : 15,
    '[20-30)' : 25,
    '[30-40)' : 35,
    '[40-50)' : 45,
    '[50-60)' : 55,
    '[60-70)' : 65,
    '[70-80)' : 75,
    '[80-90)' : 85,
    '[90-100)': 95
}

if 'age' in df.columns:
    # Preserve original for reference
    df['age_bracket'] = df['age']
    df['age'] = df['age'].map(AGE_MAP)
    unmapped = df['age'].isnull().sum()
    if unmapped > 0:
        df['age'].fillna(df['age'].median(), inplace=True)
        print(f'⚠️  {unmapped} age values unmapped → filled with median')
    print(f'Age range after conversion: {df["age"].min()} – {df["age"].max()}')
    record(9, 'age: bracket string → numeric midpoint; original kept in age_bracket')

## 8. Step 6 — Admission/Discharge Code → Labels

**Justification:** The raw IDs are numeric codes that are meaningless without the ICD reference. Mapping them to readable labels improves Tableau dashboard usability and report clarity.

In [ ]:
# ── Admission Type mapping ────────────────────────────────────────────────────
ADMISSION_TYPE_MAP = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: 'NULL',
    7: 'Trauma Center',
    8: 'Not Mapped'
}

DISCHARGE_DISPOSITION_MAP = {
    1 : 'Discharged to Home',
    2 : 'Discharged/Transferred to SNF',
    3 : 'Discharged/Transferred to SNF',
    4 : 'Discharged/Transferred to ICF',
    5 : 'Discharged/Transferred to Another Facility',
    6 : 'Discharged/Transferred to Home with Home Health',
    7 : 'Left AMA',
    8 : 'Discharged/Transferred to Home IV',
    9 : 'Admitted as Inpatient',
    10: 'Neonate/Transferred',
    12: 'Still Patient',
    15: 'Swing Bed',
    16: 'Outpatient',
    17: 'Outpatient',
    18: 'NULL',
    22: 'Rehab Facility',
    23: 'Long-term Care',
    24: 'Long-term Care',
    25: 'Not Mapped',
    26: 'Unknown',
    27: 'Inpatient Rehabilitation',
    28: 'Inpatient Rehabilitation',
    29: 'Outpatient Rehabilitation',
    30: 'Outpatient Rehabilitation'
}

ADMISSION_SOURCE_MAP = {
    1 : 'Physician Referral',
    2 : 'Clinic Referral',
    3 : 'HMO Referral',
    4 : 'Transfer from Hospital',
    5 : 'Transfer from SNF',
    6 : 'Transfer from Another Health Care Facility',
    7 : 'Emergency Room',
    8 : 'Court/Law Enforcement',
    9 : 'Not Available',
    10: 'Transfer from Critical Access Hospital',
    11: 'Normal Delivery',
    12: 'Premature Delivery',
    13: 'Sick Baby',
    14: 'Extramural Birth',
    15: 'Not Available',
    17: 'NULL',
    18: 'Transfer from Another Home Health Agency',
    19: 'Readmission to Same Home Health Agency',
    20: 'Not Mapped',
    21: 'Unknown',
    22: 'Transfer from Outpatient Surgery',
    23: 'Born Inside this Hospital',
    24: 'Born Outside this Hospital',
    25: 'Transfer from Ambulatory Surgery Center',
    26: 'Transfer from Hospice'
}

if 'admission_type_id' in df.columns:
    df['admission_type'] = df['admission_type_id'].map(ADMISSION_TYPE_MAP).fillna('Other')
    print('✅ admission_type_id → admission_type')
    record(10, 'admission_type_id → human-readable admission_type label')

if 'discharge_disposition_id' in df.columns:
    df['discharge_disposition'] = df['discharge_disposition_id'].map(DISCHARGE_DISPOSITION_MAP).fillna('Other')
    print('✅ discharge_disposition_id → discharge_disposition')
    record(11, 'discharge_disposition_id → human-readable discharge_disposition label')

if 'admission_source_id' in df.columns:
    df['admission_source'] = df['admission_source_id'].map(ADMISSION_SOURCE_MAP).fillna('Other')
    print('✅ admission_source_id → admission_source')
    record(12, 'admission_source_id → human-readable admission_source label')

## 9. Step 7 — ICD-9 Diagnosis Grouping

**Justification:** Raw ICD-9 codes like `"250.01"` are clinically precise but not useful for dashboarding or business analysis. Grouping them into 10 major disease categories enables meaningful trend analysis in Tableau.

In [ ]:
# ── ICD-9 → Disease Category ──────────────────────────────────────────────────
def icd9_to_category(code):
    """Map ICD-9 code string to a disease category label."""
    try:
        code = str(code).strip()
        # V & E codes
        if code.startswith('V') or code.startswith('E'):
            return 'External/Supplementary'
        num = float(code)
        if 250 <= num < 251:
            return 'Diabetes'
        elif 390 <= num < 460 or (num == 785):
            return 'Circulatory'
        elif 460 <= num < 520 or (num == 786):
            return 'Respiratory'
        elif 520 <= num < 580 or (num == 787):
            return 'Digestive'
        elif 800 <= num < 1000:
            return 'Injury/Poisoning'
        elif 710 <= num < 740:
            return 'Musculoskeletal'
        elif 580 <= num < 630 or (num == 788):
            return 'Genitourinary'
        elif 140 <= num < 240:
            return 'Neoplasms'
        elif 290 <= num < 320:
            return 'Mental Disorders'
        else:
            return 'Other'
    except (ValueError, TypeError):
        return 'Unknown'

for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    if diag_col in df.columns:
        new_col = diag_col.replace('diag_', 'diag_cat_')
        df[new_col] = df[diag_col].apply(icd9_to_category)
        print(f'{diag_col} → {new_col}: {df[new_col].value_counts().to_dict()}')

record(13, 'diag_1/2/3: ICD-9 codes → grouped disease category columns (diag_cat_1/2/3)')

## 10. Step 8 — Medication Column Encoding

**Justification:** Medication columns use ordinal text values (`No`, `Steady`, `Down`, `Up`). Converting to numeric ordinal codes enables correlation analysis and regression modelling.

In [ ]:
# ── Medication ordinal encoding ───────────────────────────────────────────────
MED_ORDINAL_MAP = {'No': 0, 'Steady': 1, 'Down': 2, 'Up': 3}

MED_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol',
    'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

encoded_count = 0
for col in MED_COLS:
    if col in df.columns:
        df[col + '_code'] = df[col].map(MED_ORDINAL_MAP).fillna(0).astype(int)
        encoded_count += 1

print(f'✅ {encoded_count} medication columns encoded with ordinal codes (_code suffix).')
record(14, f'{encoded_count} medication columns: text → ordinal int (_code suffix; 0=No, 1=Steady, 2=Down, 3=Up)')

# Total medication changes (proxy for treatment intensity)
med_code_cols = [c for c in df.columns if c.endswith('_code')]
df['total_med_changes'] = (df[med_code_cols] > 1).sum(axis=1)  # count of 'Up' or 'Down'
df['num_medications_active'] = (df[med_code_cols] >= 1).sum(axis=1)  # count of active meds
print(f'✅ total_med_changes and num_medications_active computed.')
record(15, 'Derived: total_med_changes (count of Up/Down), num_medications_active (count of any med active)')

## 11. Step 9 — Lab Results Encoding

**Justification:** `max_glu_serum` and `A1Cresult` use text categories. Encoding them numerically enables correlation with readmission outcomes.

In [ ]:
# ── Lab result ordinal encoding ───────────────────────────────────────────────
GLU_SERUM_MAP  = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
A1C_MAP        = {'None': 0, 'Norm': 1, '>7'  : 2, '>8'  : 3}

if 'max_glu_serum' in df.columns:
    df['max_glu_serum_code'] = df['max_glu_serum'].map(GLU_SERUM_MAP).fillna(0).astype(int)
    print('✅ max_glu_serum → max_glu_serum_code')
    record(16, 'max_glu_serum: text → ordinal code (0=None, 1=Norm, 2=>200, 3=>300)')

if 'A1Cresult' in df.columns:
    df['A1Cresult_code'] = df['A1Cresult'].map(A1C_MAP).fillna(0).astype(int)
    print('✅ A1Cresult → A1Cresult_code')
    record(17, 'A1Cresult: text → ordinal code (0=None, 1=Norm, 2=>7, 3=>8)')

## 12. Step 10 — Target Variable Engineering

**Justification:** The business question focuses on **early readmission** (within 30 days). Creating a binary `readmitted_30day` column simplifies KPI reporting and enables logistic regression in Notebook 04.

In [ ]:
# ── Readmission binary flag ───────────────────────────────────────────────────
if 'readmitted' in df.columns:
    # Binary: 1 = readmitted within 30 days, 0 = otherwise
    df['readmitted_30day'] = (df['readmitted'] == '<30').astype(int)

    # Three-class numeric: 0=NO, 1=>30 days, 2=<30 days
    READMIT_MAP = {'NO': 0, '>30': 1, '<30': 2}
    df['readmitted_code'] = df['readmitted'].map(READMIT_MAP).fillna(0).astype(int)

    rate_30 = df['readmitted_30day'].mean() * 100
    print(f'30-day readmission rate: {rate_30:.2f}%')
    print(df['readmitted'].value_counts())
    record(18, f'readmitted → readmitted_30day (binary) and readmitted_code (0=NO, 1=>30, 2=<30). 30-day rate = {rate_30:.1f}%')

## 13. Step 11 — Derived KPI Columns

**Justification:** Pre-computing KPI-ready columns in the ETL stage keeps analysis notebooks clean and ensures consistent definitions across all downstream work.

In [ ]:
# ── KPI-ready derived columns ─────────────────────────────────────────────────

# Hospital utilization intensity (procedures + lab tests per day)
if all(c in df.columns for c in ['num_lab_procedures', 'num_procedures', 'time_in_hospital']):
    df['procedures_per_day'] = (
        (df['num_lab_procedures'] + df['num_procedures']) / df['time_in_hospital'].replace(0, 1)
    ).round(2)
    print('✅ procedures_per_day computed')
    record(19, 'Derived: procedures_per_day = (num_lab_procedures + num_procedures) / time_in_hospital')

# High medication burden flag
if 'num_medications' in df.columns:
    med_75th = df['num_medications'].quantile(0.75)
    df['high_med_burden'] = (df['num_medications'] >= med_75th).astype(int)
    print(f'✅ high_med_burden flag: threshold = {med_75th:.0f} medications (75th percentile)')
    record(20, f'Derived: high_med_burden = 1 if num_medications >= {med_75th:.0f} (75th pctile)')

# Polypharmacy flag (≥5 active medications — clinical standard)
if 'num_medications_active' in df.columns:
    df['polypharmacy'] = (df['num_medications_active'] >= 5).astype(int)
    print('✅ polypharmacy flag: num_medications_active >= 5')
    record(21, 'Derived: polypharmacy = 1 if num_medications_active >= 5 (clinical polypharmacy threshold)')

# Prior hospitalisation burden
if all(c in df.columns for c in ['number_outpatient', 'number_emergency', 'number_inpatient']):
    df['prior_visits_total'] = (
        df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
    )
    print('✅ prior_visits_total computed')
    record(22, 'Derived: prior_visits_total = number_outpatient + number_emergency + number_inpatient')

## 14. Step 12 — Standardise Column Names

**Justification:** Lowercase, snake_case column names are required by Tableau's auto-field generator and prevent case-sensitivity bugs in Python.

In [ ]:
# ── Column name standardisation ───────────────────────────────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

print('✅ Column names standardised to lowercase snake_case.')
print(f'   Total columns now: {len(df.columns)}')
record(23, 'All column names → lowercase snake_case')

## 15. Final Quality Checks

In [ ]:
# ── Post-cleaning audit ───────────────────────────────────────────────────────
print('=' * 60)
print('  FINAL QUALITY AUDIT')
print('=' * 60)

print(f'\nShape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Rows removed   : {raw_rows - df.shape[0]:,}  ({(1 - df.shape[0]/raw_rows)*100:.1f}% of original)')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

print(f'\nData types:')
print(df.dtypes.value_counts())

print(f'\nCleaned column list:')
for i, c in enumerate(df.columns, 1):
    print(f'   {i:02d}. {c}')

In [ ]:
# ── Before vs After visualisation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Missing values — should be zero
null_summary = df.isnull().mean() * 100
null_summary = null_summary[null_summary > 0]

if len(null_summary) == 0:
    axes[0].text(0.5, 0.5, '✅ Zero Missing Values\nAfter Cleaning',
                 ha='center', va='center', fontsize=14,
                 color='green', fontweight='bold', transform=axes[0].transAxes)
    axes[0].set_title('Missing Values — Post-Cleaning', fontsize=12)
    axes[0].axis('off')
else:
    null_summary.plot(kind='barh', ax=axes[0], color='#FF7043')
    axes[0].set_title('Remaining Missing Values (%)', fontsize=12)

# Readmission distribution
if 'readmitted' in df.columns:
    readmit_data = df['readmitted'].value_counts()
    axes[1].pie(readmit_data, labels=readmit_data.index,
                autopct='%1.1f%%', colors=['#42A5F5', '#EF5350', '#66BB6A'],
                startangle=90)
    axes[1].set_title('Readmission Distribution — Cleaned', fontsize=12)

plt.suptitle('Post-Cleaning Quality Check', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/02_post_cleaning_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/processed/02_post_cleaning_check.png')

## 16. Export Cleaned Dataset

In [ ]:
# ── Write cleaned CSV ─────────────────────────────────────────────────────────
df.to_csv(PROCESSED_PATH, index=False)
file_size = os.path.getsize(PROCESSED_PATH) / 1_000_000

record(24, f'Cleaned dataset exported to {PROCESSED_PATH} | {df.shape[0]:,} rows × {df.shape[1]} cols | {file_size:.2f} MB')

print(f'\n✅ Cleaned dataset saved to: {PROCESSED_PATH}')
print(f'   Size: {file_size:.2f} MB')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# ── Write transformation log ──────────────────────────────────────────────────
with open(LOG_PATH, 'w') as f:
    f.write('NST DVA Capstone 2 — ETL Cleaning Log\n')
    f.write('Notebook: 02_cleaning.ipynb\n')
    f.write('Dataset : Diabetic Patients (Healthcare)\n')
    f.write('=' * 60 + '\n\n')
    for entry in log:
        f.write(entry + '\n')
    f.write(f'\nTotal transformations logged: {len(log)}\n')

print(f'✅ Transformation log saved to: {LOG_PATH}')
print(f'   Total steps logged: {len(log)}')

print('\n' + '=' * 60)
print('  CLEANING COMPLETE — Hand off to: notebooks/03_eda.ipynb')
print('=' * 60)

---
## Summary of All Cleaning Steps

| Step | Action | Justification |
|------|--------|---------------|
| 1 | Dropped zero-variance & near-empty cols | weight (97% missing), examide, citoglipton carry no analytical value |
| 2 | Removed expired/hospice discharges | Readmission not applicable to deceased patients |
| 3 | Kept first encounter per patient | Prevents data leakage; encounter-level deduplication |
| 4 | Imputed missing values | race→mode; payer_code/medical_specialty→'Unknown'; diag→'0'; numerics→median |
| 5 | Age bracket → numeric midpoint | Enables correlation and regression analysis |
| 6 | Admission/discharge IDs → labels | Human-readable labels for Tableau dashboards |
| 7 | ICD-9 codes → disease categories | Groups granular codes into 10 business-relevant categories |
| 8 | Medication text → ordinal codes | Enables correlation and statistical modelling |
| 9 | Lab results → ordinal codes | Enables correlation with readmission outcomes |
| 10 | Created readmitted_30day binary | Core KPI for 30-day readmission analysis |
| 11 | Derived KPI columns | procedures_per_day, high_med_burden, polypharmacy, prior_visits_total |
| 12 | Standardised column names | lowercase snake_case for consistency |

**Output:** `data/processed/diabetic_data_clean.csv`